In [ ]:
import pandas as pd
import numpy as np
import matplotlib as plt
import seaborn as sns
import networkx as nx

In [75]:
stops_df = pd.read_csv('gtfs/stops.txt')
platforms_df = stops_df[stops_df['parent_station'].notna()]

G = nx.Graph()
for index, row in platforms_df.iterrows():
    G.add_node(row['stop_id'], name=row['stop_name'], lat=row['stop_lat'], lon=row['stop_lon'])

print(f"Transit graph initialized with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.")

Transit graph initialized with 992 nodes and 0 edges.


In [76]:
# parent to child mapping
parent_to_children = platforms_df.groupby('parent_station')['stop_id'].apply(list).to_dict()

In [77]:
transfers_df = pd.read_csv('gtfs/transfers.txt')
walking_transfers = transfers_df[transfers_df['transfer_type'] == 2]

for index, row in walking_transfers.iterrows():
    parent_from = row['from_stop_id']
    parent_to = row['to_stop_id']
    transfer_time = row['min_transfer_time']
    
    children_from = parent_to_children.get(parent_from, [])
    children_to = parent_to_children.get(parent_to, [])
    
    for c_from in children_from:
        for c_to in children_to:
            
            # prevent self loops
            if c_from != c_to:
                
                if G.has_node(c_from) and G.has_node(c_to):
                    G.add_edge(
                        c_from, 
                        c_to, 
                        weight=transfer_time,
                        edge_type='walking_transfer'
                    )

print(f"Transit graph initialized with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.")

Transit graph initialized with 992 nodes and 763 edges.


In [79]:
def time_to_seconds(time_str):
    if pd.isna(time_str):
        return None
    
    h, m, s = map(int, time_str.split(':'))
    return h * 3600 + m * 60 + s



In [80]:
trips_df = pd.read_csv('gtfs/trips.txt')
stop_times_df = pd.read_csv('gtfs/stop_times.txt')

weekday_trips = trips_df[trips_df['service_id'].str.contains('Weekday', na=False, case=False)]

merged_df = stop_times_df.merge(weekday_trips[['trip_id', 'route_id']], on='trip_id', how='inner')

merged_df['arrival_sec'] = merged_df['arrival_time'].apply(time_to_seconds)
merged_df['departure_sec'] = merged_df['departure_time'].apply(time_to_seconds)

In [81]:
merged_df = merged_df.sort_values(by=['trip_id', 'stop_sequence'])

merged_df['next_stop_id'] = merged_df['stop_id'].shift(-1)
merged_df['next_trip_id'] = merged_df['trip_id'].shift(-1)
merged_df['next_arrival_sec'] = merged_df['arrival_sec'].shift(-1)

valid_edges = merged_df[merged_df['trip_id'] == merged_df['next_trip_id']].copy()

valid_edges['travel_time'] = valid_edges['next_arrival_sec'] - valid_edges['departure_sec']

valid_edges = valid_edges[(valid_edges['travel_time'] > 0) & (valid_edges['travel_time'] < 3600)]

In [82]:
edge_summary = valid_edges.groupby(['stop_id', 'next_stop_id']).agg(
    travel_time=('travel_time', 'mean'), routes=('route_id', lambda x: list(set(x))) # Creates a list, e.g., ['1', '2']
).reset_index()

for index, row in edge_summary.iterrows():
    
    if G.has_node(row['stop_id']) and G.has_node(row['next_stop_id']):
        G.add_edge(
            row['stop_id'], 
            row['next_stop_id'], 
            weight=row['travel_time'],
            edge_type='subway_transit',
            routes=row['routes']
        )
        
print(f"Transit graph initialized with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.")

Transit graph initialized with 992 nodes and 1900 edges.


In [87]:
print(f"Number of nodes in the transit graph: {G.number_of_nodes()}")
print(f"Number of edges in the transit graph: {G.number_of_edges()}")

Number of nodes in the transit graph: 992
Number of edges in the transit graph: 1900


In [84]:
import networkx as nx

start_node = '101S'
end_node = '120S'

try:
    # Calculate the shortest path using ONLY the edge weights (time)
    path = nx.shortest_path(G, source=start_node, target=end_node, weight='weight')
    total_time = nx.shortest_path_length(G, source=start_node, target=end_node, weight='weight')
    
    print(f"SUCCESS! Path found: {path}")
    print(f"Total commute time: {total_time / 60:.1f} minutes")
    
except nx.NetworkXNoPath:
    print("FAIL: The graph is broken. There is no path between these two stops.")
except nx.NodeNotFound as e:
    print(f"FAIL: {e}")

SUCCESS! Path found: ['101S', '103S', '104S', '106S', '107S', '108S', '109S', '110S', '111S', '112S', '113S', '114S', '115S', '116S', '117S', '118S', '119S', '120S']
Total commute time: 26.6 minutes


In [85]:
def print_directions(G, path):
    print("📍 ROUTE ITINERARY:")
    print("-------------------")
    
    current_leg_type = None
    current_route = None
    stops_count = 0
    leg_time = 0
    start_station_name = G.nodes[path[0]]['name']
    
    for i in range(len(path) - 1):
        u = path[i]
        v = path[i+1]
        edge_data = G.get_edge_data(u, v)
        
        edge_type = edge_data['edge_type']
        weight = edge_data['weight']
        
        # Scenario A: We are walking
        if edge_type == 'walking_transfer':
            # If we were previously on a train, print that train leg before we walk
            if current_leg_type == 'subway_transit':
                print(f"🚆 Take the {current_route} train for {stops_count} stops ({leg_time/60:.1f} mins)")
                print(f"   ↳ Get off at {G.nodes[u]['name']}")
                
            print(f"🚶 Walk to {G.nodes[v]['name']} ({weight/60:.1f} mins)")
            
            # Reset trackers for the next leg
            current_leg_type = 'walking_transfer'
            stops_count = 0
            leg_time = 0
            start_station_name = G.nodes[v]['name']
            
        # Scenario B: We are on a subway
        elif edge_type == 'subway_transit':
            available_routes = edge_data['routes']
            
            # If we are just starting, or switching from walking, pick the first available train
            if current_leg_type != 'subway_transit' or current_route not in available_routes:
                # If we were on a DIFFERENT train before this, print it out
                if current_leg_type == 'subway_transit':
                    print(f"🚆 Take the {current_route} train for {stops_count} stops ({leg_time/60:.1f} mins)")
                    print(f"   ↳ Transfer at {G.nodes[u]['name']}")
                else:
                    print(f"🟢 Board at {start_station_name}")
                
                # Start the new train leg
                current_route = available_routes[0] # Pick the first valid train line
                current_leg_type = 'subway_transit'
                stops_count = 0
                leg_time = 0
                
            stops_count += 1
            leg_time += weight

    # Print the final leg of the journey when the loop finishes
    if current_leg_type == 'subway_transit':
        print(f"🚆 Take the {current_route} train for {stops_count} stops ({leg_time/60:.1f} mins)")
    
    print(f"🏁 Arrive at {G.nodes[path[-1]]['name']}")
    print("-------------------")

In [90]:
start_node = '101S'
end_node = '127N'
path = nx.shortest_path(G, source=start_node, target=end_node, weight='weight')

print_directions(G, path)

📍 ROUTE ITINERARY:
-------------------
🟢 Board at Van Cortlandt Park-242 St
🚆 Take the 1 train for 17 stops (26.6 mins)
   ↳ Transfer at 96 St
🚆 Take the 3 train for 1 stops (2.9 mins)
   ↳ Get off at 72 St
🚶 Walk to 72 St (0.0 mins)
🟢 Board at 72 St
🚆 Take the 3 train for 1 stops (4.0 mins)
🏁 Arrive at Times Sq-42 St
-------------------
